# 19. Hybrid Search + Reranking

**Complexity:** 3/5

## Overview

**Hybrid search** combines keyword retrieval (BM25) with dense vector retrieval, and a **cross-encoder reranker** re-scores the fused candidates before they reach the LLM. This two-stage pattern (cheap recall, then precise ranking) is the default retrieval stack of most production RAG systems.

### The Problem

Each retriever has a blind spot:
- **Dense (embeddings)** matches meaning, but blurs exact tokens: API names (`max_retries`), error codes, IDs and rare terms often land far from the query in vector space
- **BM25 (keywords)** matches exact tokens, but misses paraphrases: "limit requests per second" never matches `rate_limiter`
- **Bi-encoders** embed query and document separately, so they cannot model fine-grained interactions between the two

### The Solution

1. **Hybrid retrieval**: run BM25 and dense retrieval in parallel and merge the rankings with weighted **Reciprocal Rank Fusion** (RRF, the same algorithm as notebook 13, here fusing two *retrievers* instead of multiple *queries*)
2. **Over-fetch**: take more candidates than you need (e.g. 20)
3. **Rerank**: a cross-encoder reads each `(query, document)` pair together and outputs a relevance score; keep the top 4

### Bi-encoder vs Cross-encoder

```
Bi-encoder (retrieval)                Cross-encoder (reranking)

 query ──► [encoder] ──► vec ─┐        (query, doc) ──► [encoder] ──► score
                               ├─ cos
 doc   ──► [encoder] ──► vec ─┘

 + docs pre-embedded once              + sees query and doc together (more accurate)
 + scales to millions                  - one forward pass per pair (slow)
```

That is why the cross-encoder is only applied to the few candidates that survived the first stage.

### Pipeline

```
                ┌─► BM25 (top 10) ──┐
Query ──────────┤                   ├─► RRF fusion ─► ~20 candidates ─► Cross-encoder ─► top 4 ─► LLM
                └─► Dense (top 10) ─┘
```

### When to Use

**Use hybrid + reranking when:**
- Queries mix natural language with identifiers (code, product SKUs, legal article numbers)
- The corpus has domain jargon the embedding model has never seen
- Precision of the top few chunks matters more than a few hundred ms of latency

**Skip it when:**
- Queries are purely conversational and the corpus is small and generic
- Latency budget is very tight and there is no GPU for the reranker

### Trade-offs

| Aspect | Hybrid | + Reranking |
|--------|--------|-------------|
| **Indexing cost** | BM25 index is cheap (in memory) | None |
| **Query latency** | ~Same as dense | +1-4 s on CPU for 20 candidates (MiniLM), ~5x more with larger models |
| **Query cost** | No extra LLM calls | No LLM calls (local model) |
| **Quality** | Better recall on exact terms | Better precision in the top-k |


## 1. Setup and Imports

In [ ]:
import sys
import time
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(str(Path("../..").resolve()))

import pandas as pd
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from shared.config import (
    verify_api_key,
    DEFAULT_MODEL,
    DEFAULT_RERANKER_MODEL,
    DEFAULT_TEMPERATURE,
    OPENAI_EMBEDDING_MODEL,
    VECTOR_STORE_DIR,
)
from shared.loaders import load_and_split
from shared.prompts import RAG_PROMPT_TEMPLATE
from shared.retrievers import (
    build_bm25_retriever,
    build_hybrid_retriever,
    build_reranking_retriever,
)
from shared.utils import (
    format_docs,
    print_section_header,
    load_vector_store,
    save_vector_store,
)

# Verify API key
verify_api_key()

print("All imports successful")
print(f"Using model: {DEFAULT_MODEL}")
print(f"Using embeddings: {OPENAI_EMBEDDING_MODEL}")
print(f"Using reranker: {DEFAULT_RERANKER_MODEL}")

## 2. Load Documents and Create Vector Store

BM25 and the vector store must index **the same chunks**: RRF matches documents across the two rankings by content. BM25 has no persistent index, so we always rebuild it from `docs` (it takes milliseconds).

In [ ]:
print_section_header("Loading Documents and Vector Store")

# Load and split documents (returns tuple: original_docs, chunks)
_, docs = load_and_split(
    chunk_size=1000,
    chunk_overlap=200,
)

print(f"\nLoaded {len(docs)} chunks")

# Initialize embeddings
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# Load or create vector store (path keyed by embedding model)
store_path = VECTOR_STORE_DIR / f"hybrid_rag__{OPENAI_EMBEDDING_MODEL}"
vectorstore = load_vector_store(store_path, embeddings)

if vectorstore is None:
    print("\nCreating vector store...")
    vectorstore = FAISS.from_documents(docs, embeddings)
    save_vector_store(vectorstore, store_path)
    print("Vector store created and saved")
else:
    print("Loaded existing vector store")

## 3. Build the Retrievers

We compare four retrievers, all returning the **top 4** documents to the LLM:

| Retriever | First stage | Second stage |
|-----------|-------------|--------------|
| **Dense** | FAISS, k=4 | - |
| **BM25** | BM25, k=4 | - |
| **Hybrid** | BM25 + FAISS, k=4 each, fused with RRF | truncate to 4 |
| **Hybrid + Rerank** | BM25 + FAISS, k=10 each, fused with RRF | cross-encoder, top 4 |

BM25 tokenization matters for code-heavy docs: `build_bm25_retriever()` lowercases and splits on non-word characters (`bm25_tokenize`), because LangChain's default splits on whitespace only and `max_retries=2,` would never match the query `max_retries`.

`build_hybrid_retriever()` and `build_reranking_retriever()` live in `shared/retrievers.py` and wrap LangChain's `EnsembleRetriever` and `ContextualCompressionRetriever` + `CrossEncoderReranker` (from `langchain-classic` in LangChain 1.x).

In [ ]:
print_section_header("Building Retrievers")

K = 4
FETCH_K = 10  # Candidates per retriever before reranking

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": K})
bm25_retriever = build_bm25_retriever(docs, k=K)
hybrid_retriever = build_hybrid_retriever(docs, vectorstore, k=K, bm25_weight=0.5)

# Load the cross-encoder once and reuse it (first run downloads ~90 MB)
print(f"Loading cross-encoder {DEFAULT_RERANKER_MODEL}...")
cross_encoder = HuggingFaceCrossEncoder(model_name=DEFAULT_RERANKER_MODEL)

rerank_retriever = build_reranking_retriever(
    build_hybrid_retriever(docs, vectorstore, k=FETCH_K, bm25_weight=0.5),
    top_n=K,
    cross_encoder=cross_encoder,
)

retrievers = {
    "Dense": dense_retriever,
    "BM25": bm25_retriever,
    "Hybrid": hybrid_retriever,
    "Hybrid + Rerank": rerank_retriever,
}

print("\nRetrievers ready:")
for name in retrievers:
    print(f"  • {name}")

## 4. Side-by-Side: Exact Terms vs Paraphrases

Two queries that stress opposite blind spots:
1. An exact identifier (`batch_as_completed`): BM25 territory
2. A paraphrase that shares no keywords with the answer: dense territory

In [ ]:
def show_results(query: str, retrievers: dict, k: int = K) -> None:
    # Print the top-k preview of each retriever for one query
    print("\n" + "=" * 80)
    print(f"Query: {query}")
    print("=" * 80)
    for name, retriever in retrievers.items():
        results = retriever.invoke(query)[:k]
        print(f"\n[{name}]")
        for i, doc in enumerate(results, 1):
            preview = " ".join(doc.page_content.split())[:110]
            print(f"  {i}. {preview}...")


show_results("batch_as_completed", retrievers)
show_results(
    "How do I limit how many requests per second my app sends to the model provider?",
    retrievers,
)

## 5. Quantitative Comparison

A small labeled set: each query is paired with **markers**, strings that only occur in the chunks that answer it (silver labels, checked against the corpus). A retrieved chunk is relevant if it contains any marker.

We report two metrics over the top 4:
- **Hit@4**: fraction of queries with at least one relevant chunk in the top 4
- **MRR@4**: mean of `1 / rank` of the first relevant chunk (0 if none), rewards putting it first

In [ ]:
EVAL_SET = [
    # Exact-term queries (identifiers)
    {"type": "keyword", "query": "max_retries", "markers": ["max_retries"]},
    {"type": "keyword", "query": "rate_limiter parameter", "markers": ["rate_limiter"]},
    {"type": "keyword", "query": "SummarizationMiddleware", "markers": ["SummarizationMiddleware"]},
    {"type": "keyword", "query": "batch_as_completed", "markers": ["batch_as_completed"]},
    {"type": "keyword", "query": "reasoning_effort", "markers": ["reasoning_effort"]},
    {"type": "keyword", "query": "configurable_model", "markers": ["configurable_model"]},
    # Paraphrased queries (no shared identifiers)
    {
        "type": "semantic",
        "query": "How do I limit how many requests per second my app sends to the model provider?",
        "markers": ["rate_limiter"],
    },
    {
        "type": "semantic",
        "query": "How can I automatically condense old conversation history into a shorter version?",
        "markers": ["SummarizationMiddleware"],
    },
    {
        "type": "semantic",
        "query": "How do I make the model reply with data that follows a schema?",
        "markers": ["with_structured_output"],
    },
    {
        "type": "semantic",
        "query": "How do I get results for many inputs as soon as each one finishes?",
        "markers": ["batch_as_completed"],
    },
    {
        "type": "semantic",
        "query": "How do I drop the oldest messages so the history fits in the context window?",
        "markers": ["trim_messages", "RemoveMessage"],
    },
    {
        "type": "semantic",
        "query": "How do I switch between model providers at runtime without changing code?",
        "markers": ["configurable_model"],
    },
]


def first_relevant_rank(results, markers) -> int | None:
    # 1-based rank of the first chunk containing a marker, None if absent
    for rank, doc in enumerate(results, 1):
        if any(m in doc.page_content for m in markers):
            return rank
    return None


def evaluate(retrievers: dict, eval_set: list, k: int = K) -> pd.DataFrame:
    rows = []
    for name, retriever in retrievers.items():
        for item in eval_set:
            start = time.time()
            results = retriever.invoke(item["query"])[:k]
            latency = time.time() - start
            rank = first_relevant_rank(results, item["markers"])
            rows.append(
                {
                    "retriever": name,
                    "type": item["type"],
                    "hit": rank is not None,
                    "rr": 1 / rank if rank else 0.0,
                    "latency_ms": latency * 1000,
                }
            )
    return pd.DataFrame(rows)


print_section_header("Evaluating Retrievers")
results_df = evaluate(retrievers, EVAL_SET)

summary = (
    results_df.groupby(["retriever", "type"])[["hit", "rr"]]
    .mean()
    .unstack("type")
    .rename(columns={"hit": "Hit@4", "rr": "MRR@4"})
)
summary[("Hit@4", "overall")] = results_df.groupby("retriever")["hit"].mean()
summary[("MRR@4", "overall")] = results_df.groupby("retriever")["rr"].mean()
summary = summary.sort_index(axis=1).reindex(list(retrievers))
print(summary.round(2).to_string())

**How to read it:** BM25 should win on identifiers and dense on paraphrases; hybrid should be at least as good as the better of the two on both, and the reranker should mostly improve **MRR** (it reorders what hybrid already found, and can surface candidates from ranks 5-20).

> With 12 queries, one query moves a score by ~0.17: treat this as a sanity check, not a benchmark. Notebook 16 shows how to evaluate end-to-end answers with RAGAS.

## 6. Tuning the BM25 Weight

`bm25_weight` sets how much each ranking counts in the RRF sum. 0 = pure dense, 1 = pure BM25.

In [ ]:
print_section_header("BM25 Weight Sweep")

sweep = []
for w in [0.0, 0.25, 0.5, 0.75, 1.0]:
    retriever = build_hybrid_retriever(docs, vectorstore, k=K, bm25_weight=w)
    df = evaluate({f"w={w}": retriever}, EVAL_SET)
    sweep.append(
        {
            "bm25_weight": w,
            "Hit@4 keyword": df[df.type == "keyword"].hit.mean(),
            "Hit@4 semantic": df[df.type == "semantic"].hit.mean(),
            "MRR@4": df.rr.mean(),
        }
    )

print(pd.DataFrame(sweep).round(2).to_string(index=False))

## 7. Inside the Reranker

The cross-encoder returns a raw relevance score for each `(query, chunk)` pair. Let's look at how it reorders the hybrid candidates for a paraphrased query.

In [ ]:
print_section_header("Reranker Scores")

query = "How do I drop the oldest messages so the history fits in the context window?"
markers = ["trim_messages", "RemoveMessage"]

candidates = build_hybrid_retriever(docs, vectorstore, k=FETCH_K).invoke(query)
scores = cross_encoder.score([(query, doc.page_content) for doc in candidates])

rows = [
    {
        "hybrid_rank": i,
        "rerank_score": round(float(score), 3),
        "relevant": any(m in doc.page_content for m in markers),
        "preview": " ".join(doc.page_content.split())[:70],
    }
    for i, (doc, score) in enumerate(zip(candidates, scores), 1)
]
table = pd.DataFrame(rows).sort_values("rerank_score", ascending=False)
table.insert(0, "new_rank", range(1, len(table) + 1))

print(f"Query: {query}")
print(f"Candidates from hybrid: {len(candidates)}\n")
print(table.head(8).to_string(index=False))

## 8. Hybrid + Rerank RAG Chain

In [ ]:
print_section_header("Building RAG Chain")

llm = ChatOpenAI(model=DEFAULT_MODEL, temperature=DEFAULT_TEMPERATURE)

hybrid_rag_chain = (
    {"context": rerank_retriever | format_docs, "input": RunnablePassthrough()}
    | RAG_PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

print("Hybrid + Rerank RAG chain created")

for query in [
    "What does max_retries control and what is its default?",
    "How do I drop the oldest messages so the history fits in the context window?",
]:
    print("\n" + "=" * 80)
    print(f"Query: {query}")
    print("=" * 80)
    start = time.time()
    print(hybrid_rag_chain.invoke(query))
    print(f"\nTime: {time.time() - start:.2f}s")

## 9. Performance Metrics

In [ ]:
print_section_header("Retrieval Latency")

latency = results_df.groupby("retriever")["latency_ms"].agg(["mean", "max"]).reindex(list(retrievers))
print(latency.round(0).to_string())

print("\nNotes:")
print("  • BM25 runs in memory and is effectively free")
print("  • Dense latency is dominated by the query embedding API call")
print(f"  • Reranking scores up to {2 * FETCH_K} (query, chunk) pairs on CPU: latency grows")
print("    linearly with candidates and chunk length. Larger rerankers (BAAI/bge-reranker-base)")
print("    are ~5x slower on CPU; use a GPU or a hosted reranker for them")

## 10. Key Takeaways

### Summary

- **BM25 and dense retrieval fail on different queries**: identifiers vs paraphrases. Fusing them with RRF recovers most of both
- **RRF needs no score normalization**: BM25 scores and cosine similarities live on different scales, ranks do not
- **Reranking is a precision step**: over-fetch with a cheap retriever, then let a cross-encoder pick the best few

### Cost-Benefit Analysis

| Aspect | Impact | Notes |
|--------|--------|-------|
| **Indexing Cost** | ~Same | BM25 index built in memory from the same chunks |
| **Query Latency** | +reranking time | Scales with number of candidates |
| **Query Cost** | Same | Reranker runs locally, no extra LLM calls |
| **Retrieval Quality** | Better | Especially on identifiers and jargon |
| **Complexity** | Moderate | Two indexes to keep in sync |

### Best Practices

1. **Keep BM25 and the vector store on the same chunks**, rebuilt together
2. **Start with `bm25_weight=0.5`**, then tune on a labeled set like the one above
3. **Over-fetch 3-5x** the final k before reranking
4. **Reuse the cross-encoder instance**: loading it is the slow part
5. **Tokenize BM25 for your corpus**: split on punctuation (as `bm25_tokenize` does) so identifiers next to `=`, `(` or `,` still match

### Production Notes

- Managed vector databases (Qdrant, Weaviate, Elasticsearch/OpenSearch, pgvector + full text search) offer hybrid search natively, so you don't keep BM25 in process memory
- The default reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`, English, ~90 MB) is chosen to run on a laptop CPU. With a GPU, larger or multilingual models such as `BAAI/bge-reranker-base` or `BAAI/bge-reranker-v2-m3` are usually more accurate (set `DEFAULT_RERANKER_MODEL`)
- Hosted rerankers (Cohere Rerank, Voyage, Jina) avoid running the model yourself; they plug into the same `ContextualCompressionRetriever`

### Next Steps

- **Notebook 12**: contextual retrieval, which combines contextual chunks with this same hybrid + rerank stack
- **Notebook 20**: parent-document and multi-vector retrieval, which change *what* you index
- **Notebook 16**: evaluate the full pipeline with RAGAS